# Mirai Studio — Anime & Mangá na GPU grátis do Colab

Este notebook sobe **Automatic1111** (imagem: anime E mangá) **e** o **ComfyUI**
(vídeo: AnimateDiff, Wan 2.2, LTXV, Hyper-SD) na GPU T4 gratuita do Colab, e
imprime as URLs públicas que o Mirai Studio consome.

**ANTES DE COMEÇAR (faça isto primeiro):**
1. Menu → **Ambiente de execução → Alterar tipo de ambiente de execução → Acelerador de hardware: GPU (T4)** → Salvar.
2. Depois execute as células **na ordem, de cima para baixo**.

> **Por que A1111 para imagem:** tem `--share` nativo, que entrega uma URL
 `*.gradio.live` pública sem instalar túnel. O ComfyUI precisa de um passo
 extra de `cloudflared` — por isso ele vem DEPOIS, e só se você quer vídeo.
>
> **Segurança:** a URL pública é a sua credencial (as APIs não têm
 autenticação). Não compartilhe, não comite, e derrube o notebook quando
 terminar — ou use `--gradio-auth usuario:senha` (já incluído).

In [ ]:
# 1) Confirmar que uma GPU real está ligada (deve dizer Tesla T4).
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2) Instalar o Automatic1111 (imagem: anime e mangá).
import os
if not os.path.isdir('stable-diffusion-webui'):
    !git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui
%cd stable-diffusion-webui
!pip install -q -r requirements.txt
%cd ..

In [ ]:
# 3) Baixar os CHECKPOINTS do estilo que você quer.

SUBSTITUA as URLs pelos links diretos (.safetensors) da página de cada
modelo (Civitai: página do modelo → Download → copiar link; Hugging Face:
o link de download ao lado do arquivo).

REFERÊNCIAS DE QUALIDADE (nível dos estúdios que você citou):
#  ANIME  -> Pony Diffusion V6 XL · Illustrious (v1.0/v2.0) · Counterfeit · Animagine XL
#  MANGÁ  -> os acima + um LoRA de lineart/ink; para PB use um de 'monochrome'

CHECKPOINTS = {
    # 'ponyV6XL.safetensors':  'URL_DIRETA_AQUI',
    # 'illustrious.safetensors': 'URL_DIRETA_AQUI',
}

if not CHECKPOINTS:
    print('Preencha CHECKPOINTS com URLs diretas e execute esta célula de novo.')
else:
    import urllib.request, os
    dest = 'stable-diffusion-webui/models/Stable-diffusion'
    os.makedirs(dest, exist_ok=True)
    for name, url in CHECKPOINTS.items():
        target = os.path.join(dest, name)
        print('baixando', name, '...')
        urllib.request.urlretrieve(url, target)
        print('  ok ->', target, os.path.getsize(target) // 1024 // 1024, 'MB')
    !ls -la {dest}

In [ ]:
# 4) INICIAR o servidor de IMAGEM (A1111) com URL pública.
#    Deixe esta célula rodando: ela imprime o link *.gradio.live que o
#    Mirai Studio precisa. --api liga a API HTTP; --gradio-auth põe senha.
%cd stable-diffusion-webui
!python launch.py --api --share --no-download-sd-model --xformers --gradio-auth mirai:mirai

## 5) Conectar a IMAGEM no Mirai Studio

Quando a célula 4 imprimir `Running on public URL: https://xxxx.gradio.live`:

1. **AI Marketplace** → **"Your Own GPU (Automatic1111)"** → *Usar provedor*.
2. **Settings → Providers → image → "Your GPU"**: cole a URL, marque **"allow a remote GPU"**.
3. Escolha o **checkpoint** na lista (o app lê o que você instalou).
4. Storyboard → **Generate (AI)**.

O prompt já carrega o contrato de mídia (2D JAPANESE ANIME CEL-ANIMATION)
e a referência de estúdio (CloverWorks/My Dress-Up Darling, White Fox/Re:Zero,
Studio Deen/KonoSuba, Toei, Kyoto Animation, A-1 Pictures, Wit, Ufotable) + o
`negative_prompt` que exclui photorealistic/3D/live action pelo nome.

---
## 6) VÍDEO (opcional) — ComfyUI com AnimateDiff / Wan 2.2 / LTXV

O ComfyUI **não** tem `--share`. São duas células: instalar + túnel.
Se você só quer imagem/keyframe, PARE aqui.

In [ ]:
# 6a) Instalar o ComfyUI + os nós de VÍDEO.
!git clone https://github.com/comfyanonymous/ComfyUI 2>/dev/null || echo 'ComfyUI já existe'
%cd ComfyUI
!pip install -q -r requirements.txt
!git clone https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved custom_nodes/ComfyUI-AnimateDiff-Evolved
!pip install -q -r custom_nodes/ComfyUI-AnimateDiff-Evolved/requirements.txt
!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper custom_nodes/ComfyUI-WanVideoWrapper
!pip install -q -r custom_nodes/ComfyUI-WanVideoWrapper/requirements.txt
!git clone https://github.com/alisson-anjos/ComfyUI-HyperSDXL custom_nodes/ComfyUI-HyperSDXL 2>/dev/null || true
%cd ..
print('ComfyUI + nós de vídeo instalados')

In [ ]:
# 6b) Baixar os modelos de VÍDEO (AnimateDiff / Wan / LTXV).
SUBSTITUA pelas URLs diretas dos modelos. Tamanhos aproximados:
#  AnimateDiff motion module : ~1.7 GB
#  SDXL base                 : ~6.9 GB
#  Wan 2.2 / LTXV            : 8-20 GB (NÃO cabe na T4 grátis — use GPU maior)

VIDEO_MODELS = {
    # 'ComfyUI/models/animatediff_lightning_4step_diffusers.safetensors': 'URL_DIRETA_AQUI',
    # 'ComfyUI/models/checkpoints/sd_xl_base_1.0.safetensors': 'URL_DIRETA_AQUI',
}

if not VIDEO_MODELS:
    print('Preencha VIDEO_MODELS com URLs diretas e execute de novo.')
else:
    import urllib.request, os
    for target, url in VIDEO_MODELS.items():
        os.makedirs(os.path.dirname(target), exist_ok=True)
        urllib.request.urlretrieve(url, target)
        print('ok ->', target, os.path.getsize(target) // 1024 // 1024, 'MB')

In [ ]:
# 6c) INICIAR o ComfyUI e abrir um TÚNEL público (cloudflared, sem conta).
#    A célula imprime a URL https://*.trycloudflare.com no fim.
%cd ComfyUI
!python main.py --listen &
%cd ..
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8188

## 7) Conectar o VÍDEO

1. **AI Marketplace** → **"Your Own GPU (ComfyUI)"** → *Usar provedor*.
2. **Settings → Providers → video → "Your GPU"**: cole a URL
   `*.trycloudflare.com`, marque **"allow a remote GPU"**.
3. Cole o workflow: no ComfyUI, monte o grafo (AnimateDiff/Wan/LTXV), use
   **Save (API Format)** e cole o JSON no campo *ComfyUI workflow*.

O app injeta antes de enviar: `{{prompt}}`, `{{negative}}`, `{{seed}}` e
`{{image}}` (o frame do storyboard, que ele sobe pra você).

**A receita que trava o personagem** (é o que fez o anime do Draco viralizar):
1. **Model sheet** do personagem (Character editor → *AI sheet*) — a âncora.
2. **Keyframe** usando a model sheet como referência (img2img, denoise 0.55).
3. **Animar o keyframe** (image-to-video) — o clipe herda o design.
4. **Curtes curtos, muitos deles** (2–3 s) — é assim que anime disfarça
   movimento limitado, e é onde esses modelos são fortes.

**O que NINGUÉM faz hoje:** vídeo longo de uma chamada. Kling 3.0, Wan 2.7 e
Seedance geram 3–15 s. Um episódio de 22 min são ~160 clipes — o app calcula,
gera em sequência e costura sem perda.

**GPU esperada:** imagem 6–12 GB · AnimateDiff 8–12 GB · Wan 2.2+ 12–24 GB.
A T4 grátis (16 GB) serve para imagem e AnimateDiff; para Wan 2.2 use GPU maior.
O app verifica a VRAM do seu backend **antes** de começar (Settings → "Your GPU").